In [ ]:
!pip install -q transformers torch tqdm h5py opencv-python-headless

In [ ]:
import os, time, numpy as np, torch, torch.nn.functional as F, tarfile, cv2
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor

# Universal secret getter
def get_secret(key: str, fallback_path: str = None, default=None) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        pass
    if fallback_path and os.path.exists(fallback_path):
        return open(fallback_path).read().strip()
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    value = os.environ.get(key, default)
    if value is None:
        raise EnvironmentError(f"Secret '{key}' not found")
    return value

hf_token = get_secret("HF_TOKEN", fallback_path="/kaggle/input/my-hf-secrets/HF_TOKEN.txt")
print("✓ HF_TOKEN loaded")

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}
DATASET_BLACKLIST = {"my-hf-secrets", "kaggle-data-overlay"}

def find_and_extract_dataset():
    """Find dataset, extract tar if needed, return path with images"""
    if not INPUT_DIR.exists():
        return None
    for dataset_dir in sorted(INPUT_DIR.iterdir()):
        if not dataset_dir.is_dir():
            continue
        if dataset_dir.name in DATASET_BLACKLIST:
            continue
        tar_files = list(dataset_dir.glob("*.tar")) + list(dataset_dir.glob("*.tar.gz"))
        if tar_files:
            extract_dir = WORK_DIR / "dataset"
            extract_dir.mkdir(exist_ok=True)
            print(f"Extracting {tar_files[0].name}...")
            with tarfile.open(tar_files[0], 'r:*') as tar:
                tar.extractall(extract_dir)
            print(f"✓ Extracted to {extract_dir}")
            return str(extract_dir)
        for ext in SUPPORTED_EXTS:
            if list(dataset_dir.rglob(f"*{ext}")):
                return str(dataset_dir)
    return None

DATASET_PATH = find_and_extract_dataset()
if not DATASET_PATH:
    raise FileNotFoundError(f"No dataset found in {INPUT_DIR}")
# Sanity check the extracted dataset (catches corrupt tar / partial upload)
ext_count = sum(1 for _ in Path(DATASET_PATH).rglob("*") if _.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
if ext_count < 100:
    raise RuntimeError(f"Only {ext_count} images found — dataset may be corrupt. Aborting.")
print(f"✓ Dataset: {DATASET_PATH} ({ext_count} images)")

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
EMBEDDING_DIM = 768
OUTPUT_DIR = "/kaggle/working"

# Adaptive GPU/CPU configuration
# Primary target: T4 x2 (Kaggle free-tier)
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    if "P100" in gpu_name:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 128, 4
        print(f"GPU: {gpu_name} (fallback) | Batch: {BATCH_SIZE}")
    elif num_gpus > 1:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 448, 4
        print(f"GPU: {gpu_name} x{num_gpus} | Batch: {BATCH_SIZE}")
    else:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 256, 4
        print(f"GPU: {gpu_name} | Batch: {BATCH_SIZE}")
else:
    device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
    print("CPU mode")


In [ ]:
# Read preprocessing config from model, then build pure cv2/numpy pipeline
# (no PIL dependency — eliminates cv2→PIL conversion overhead)
hf_proc = AutoImageProcessor.from_pretrained(MODEL_ID, token=hf_token)
IMG_SIZE  = hf_proc.size.get('shortest_edge', hf_proc.size.get('height', 224))
CROP_SIZE = hf_proc.crop_size.get('height', 224) if hasattr(hf_proc, 'crop_size') and hf_proc.crop_size else IMG_SIZE
IMG_MEAN  = np.array(hf_proc.image_mean, dtype=np.float32)  # [0.485, 0.456, 0.406]
IMG_STD   = np.array(hf_proc.image_std,  dtype=np.float32)  # [0.229, 0.224, 0.225]
print(f"Preprocessing: resize={IMG_SIZE}, crop={CROP_SIZE}")
del hf_proc

class FastDataset(Dataset):
    """Pure cv2/numpy pipeline — no PIL, no torchvision overhead."""
    def __init__(self, paths, img_size, crop_size, mean, std):
        self.paths = paths
        self.img_size = img_size
        self.crop_size = crop_size
        self.mean = mean.reshape(1, 1, 3)
        self.std  = std.reshape(1, 1, 3)

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        try:
            img = cv2.imread(self.paths[i], cv2.IMREAD_COLOR)
            if img is None:
                raise ValueError("cv2 failed")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Resize shortest edge → img_size
            h, w = img.shape[:2]
            if h <= w:
                new_h = self.img_size
                new_w = int(round(w * self.img_size / h))
            else:
                new_w = self.img_size
                new_h = int(round(h * self.img_size / w))
            # INTER_AREA: fastest and best quality for downscaling
            img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

            # Center crop
            y = (new_h - self.crop_size) // 2
            x = (new_w - self.crop_size) // 2
            img = img[y:y + self.crop_size, x:x + self.crop_size]

            # Normalize: uint8 HWC → float32 CHW
            img = img.astype(np.float32) / 255.0
            img = (img - self.mean) / self.std
            img = np.ascontiguousarray(img.transpose(2, 0, 1))  # HWC → CHW
            return torch.from_numpy(img), self.paths[i]

        except Exception:
            z = np.zeros((3, self.crop_size, self.crop_size), dtype=np.float32)
            return torch.from_numpy(z), self.paths[i]


imgs = [
    (str(p.relative_to(DATASET_PATH)), str(p))
    for p in Path(DATASET_PATH).rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
]
image_ids, image_paths = [i for i, _ in imgs], [p for _, p in imgs]
print(f"Found {len(image_ids)} images")

In [ ]:
print(f"Loading {MODEL_ID}...")
t0 = time.time()

dtype = torch.float16 if device == "cuda" else torch.float32
use_amp = (device == "cuda")
if use_amp: print("  Using FP16 for speedup.")

model = AutoModel.from_pretrained(
    MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True, token=hf_token
)
torch.cuda.empty_cache()

class VisionWrapper(torch.nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.model = full_model
    def forward(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        # CLS token at position 0 — more reliable than pooler_output
        # which may use newly initialized (untrained) weights
        return outputs.last_hidden_state[:, 0]

wrapped_model = VisionWrapper(model)

# torch.compile for single GPU only (incompatible with DataParallel)
if device == "cuda" and torch.cuda.device_count() == 1:
    try:
        # "reduce-overhead" uses CUDA graphs — biggest win for inference
        wrapped_model = torch.compile(wrapped_model, mode='reduce-overhead')
        print("  ✓ Applied torch.compile (reduce-overhead)")
    except Exception as e:
        print(f"  ⚠ torch.compile skipped: {e}")

if device == "cuda" and torch.cuda.device_count() > 1:
    final_model = torch.nn.DataParallel(wrapped_model).cuda()
    print("  ✓ Applied DataParallel")
else:
    final_model = wrapped_model.to(device)

final_model.eval()
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    # Warmup pass so cuDNN picks the best kernels before timing starts
    with torch.no_grad():
        dummy = torch.randn(BATCH_SIZE, 3, CROP_SIZE, CROP_SIZE, device=device, dtype=dtype)
        for _ in range(2):
            _ = final_model(dummy)
        torch.cuda.synchronize()
    print("  ✓ cuDNN warmup done")
print(f"Model ready in {time.time()-t0:.1f}s")

In [ ]:
    loader = DataLoader(
        FastDataset(image_paths, IMG_SIZE, CROP_SIZE, IMG_MEAN, IMG_STD),
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda"),
        pin_memory_device="cuda" if device == "cuda" else "",
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=3 if NUM_WORKERS > 0 else None,
    )

    # Defer H2D + numpy: keep fp16 features on GPU, do a single cat+transfer at the end.
    # Old code did .cpu().float().numpy() per batch — that forces a CUDA sync per batch
    # (~12 syncs for 5400 images / 448 batch). Removing those syncs is the biggest win.
    gpu_feats = []
    t0 = time.time()
    n_batches = 0

    with torch.inference_mode():  # stricter than no_grad (~5% faster)
        for pixel_values, _ in loader:
            pixel_values = pixel_values.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    feat = F.normalize(final_model(pixel_values), p=2, dim=-1)
            else:
                feat = F.normalize(final_model(pixel_values), p=2, dim=-1)
            gpu_feats.append(feat)  # stay on GPU, no per-batch sync
            n_batches += 1

    # One sync + one cat + one transfer at the end
    torch.cuda.synchronize()
    sync_t = time.time() - t0
    embeddings = torch.cat(gpu_feats, dim=0).cpu().float().numpy()
    inf_time = time.time() - t0
    n_done = embeddings.shape[0]
    per_batch_ms = (inf_time * 1000 / n_batches) if n_batches else 0
    d2h_t = inf_time - sync_t
    print(f"✓ Inference: {inf_time:.1f}s ({n_done / inf_time:.1f} img/s, {n_batches} batches)")
    print(f"  Per-batch avg: {per_batch_ms:.0f}ms | cat+D2H at end: {d2h_t:.2f}s")




In [ ]:
# Save embeddings to HDF5 (CLS-token features)
import h5py

hdf5_path = OUTPUT_DIR + "/embeddings.hdf5"

with h5py.File(hdf5_path, "w") as f:
    f.create_dataset("embeddings", data=embeddings)
    f.create_dataset("image_ids", data=[s.encode("utf-8") for s in image_ids])
    f.create_dataset("image_paths", data=[s.encode("utf-8") for s in image_paths])
    f.attrs["model_id"]      = MODEL_ID
    f.attrs["feature_type"]  = "image_features"
    f.attrs["embedding_dim"] = EMBEDDING_DIM
    f.attrs["num_images"]    = len(image_ids)

print(f"[OK] Saved embeddings.hdf5 ({embeddings.nbytes / 1024**2:.1f} MB)")

# Free memory (Kaggle session can be killed if RAM > 16GB)
del embeddings
import gc; gc.collect()
if device == "cuda": torch.cuda.empty_cache()
print("[OK] Pipeline complete")
